In [1]:
pip install transformers datasets scikit-learn torch tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 48.8 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.

In [2]:
import random
import numpy as np
import torch


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

In [3]:
set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd


def load_and_split(path):
    df = pd.read_csv(path, sep=";")

    texts = df["Text"].values
    labels = df["Score"].values

    le = LabelEncoder()
    labels = le.fit_transform(labels)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=42,
        stratify=labels
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.1,
        random_state=42,
        stratify=y_train_full
    )

    print("Train:", len(X_train))
    print("Val:", len(X_val))
    print("Test:", len(X_test))

    return X_train, X_val, X_test, y_train, y_val, y_test, le

In [5]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset


MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        enc = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
from torch.utils.data import DataLoader

def build_loaders(X_train, X_val, X_test, y_train, y_val, y_test):

    train_ds = TextDataset(X_train, y_train)
    val_ds   = TextDataset(X_val, y_val)
    test_ds  = TextDataset(X_test, y_test)

    generator = torch.Generator()
    generator.manual_seed(42)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, generator=generator)
    val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False, generator=generator)
    test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, generator=generator)

    return train_loader, val_loader, test_loader

In [7]:
import torch.nn as nn
from transformers import AutoModel


class BertClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):

        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)

        return self.fc(cls)

In [ ]:
def q_temperature_loss(logits, labels, q=1.0, temperature=1.0, weights=None):

    logits = logits / temperature
    logits = torch.clamp(logits, -10, 10)

    idx = torch.arange(logits.size(0), device=logits.device)

    if abs(q - 1.0) < 1e-6:
        log_probs = torch.log_softmax(logits, dim=1)
        loss = -log_probs[idx, labels]
        return (loss * weights[labels]).mean()

    base = 1 + (1 - q) * logits
    base = torch.clamp(base, min=1e-8)

    exps = base ** (1 / (1 - q))
    probs = exps / (exps.sum(dim=1, keepdim=True) + 1e-8)

    p = probs[idx, labels]
    p = torch.clamp(p, min=1e-8)

    loss = -((p ** (1 - q) - 1) / (1 - q))

    return (loss * weights[labels]).mean()

In [ ]:
import torch
from tqdm import tqdm

def train(
    model,
    train_loader,
    val_loader,
    q,
    temperature,
    weights=None,
    epochs=10,
    patience=3
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    if weights is not None:
        weights = weights.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-5
    )

    best_val = float("inf")
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_loader):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            logits = model(
                input_ids,
                attention_mask
            )

            loss = q_temperature_loss(
                logits,
                labels,
                q=q,
                temperature=temperature,
                weights=weights
            )

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0.0

        with torch.no_grad():

            for batch in val_loader:

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = q_temperature_loss(
                    logits,
                    labels,
                    q=q,
                    temperature=temperature,
                    weights=weights
                )

                val_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        print(
            f"Epoch {epoch+1} | "
            f"Train={train_loss:.4f} | "
            f"Val={val_loss:.4f}"
        )

        if val_loss < best_val:

            best_val = val_loss
            no_improve = 0

            torch.save(
                model.state_dict(),
                "best_model.pt"
            )

            print("Saved best model")

        else:

            no_improve += 1

            print(
                f"No improvement "
                f"({no_improve}/{patience})"
            )

            if no_improve >= patience:

                print("Early stopping triggered")
                break

In [10]:
from sklearn.metrics import classification_report


def evaluate(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    preds, true = [], []

    with torch.no_grad():
        for batch in test_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            pred = torch.argmax(logits, dim=1)

            preds.extend(pred.cpu().numpy())
            true.extend(labels.cpu().numpy())

    print(classification_report(true, preds))

In [ ]:
def main(q=1.0, temperature=1.0):

    path = "/kaggle/input/datasets/librust/hw1-dataset/dataset_hw1.csv"

    X_train, X_val, X_test, y_train, y_val, y_test, le = load_and_split(path)

    train_loader, val_loader, test_loader = build_loaders(
        X_train, X_val, X_test,
        y_train, y_val, y_test
    )

    model = BertClassifier(num_classes=len(le.classes_))

    counts = np.bincount(y_train)
    total = counts.sum()

    weights = (total / counts) ** 0.75
    weights = weights / weights.mean()
    weights = torch.tensor(
        weights,
        dtype=torch.float32
    )
    
    train(
        model,
        train_loader,
        val_loader,
        q=q,
        temperature=temperature,
        weights=weights
    )

    model.load_state_dict(torch.load("best_model.pt"))

    evaluate(model, test_loader)

In [12]:
qs = [0.4, 0.6, 0.8, 1.0]
temps = [0.5, 1.0, 1.5, 2.0]

In [13]:
main()

Train: 20464
Val: 2274
Test: 5685


pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  0%|          | 0/1279 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

Epoch 4 | Train=0.4242 | Val=0.5827
No improvement (1/3)


100%|██████████| 1279/1279 [05:26<00:00,  3.92it/s]


Epoch 5 | Train=0.3774 | Val=0.5824
No improvement (2/3)


100%|██████████| 1279/1279 [05:26<00:00,  3.92it/s]


Epoch 6 | Train=0.3313 | Val=0.5984
No improvement (3/3)
Early stopping triggered
              precision    recall  f1-score   support

           0       0.64      0.64      0.64       526
           1       0.40      0.33      0.36       300
           2       0.32      0.62      0.42       428
           3       0.39      0.42      0.40       794
           4       0.91      0.80      0.85      3637

    accuracy                           0.70      5685
   macro avg       0.53      0.56      0.54      5685
weighted avg       0.74      0.70      0.71      5685



In [ ]:
for q in qs:
    for t in temps:
        print(f"==============\nq = {q}, T = {t}\n")
        main(q,t)

q = 0.4, T = 0.5

Train: 20464
Val: 2274
Test: 5685


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 1279/1279 [05:26<00:00,  3.92it/s]


Epoch 1 | Train=0.4092 | Val=0.3502
Saved best model


100%|██████████| 1279/1279 [05:25<00:00,  3.93it/s]


...

100%|██████████| 1279/1279 [05:27<00:00,  3.90it/s]


Epoch 1 | Train=0.6751 | Val=0.5831
Saved best model


100%|██████████| 1279/1279 [05:26<00:00,  3.92it/s]


Epoch 2 | Train=0.5584 | Val=0.5528
Saved best model


100%|██████████| 1279/1279 [05:26<00:00,  3.92it/s]
